In [1]:
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
import os

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Paths
DATA_DIR = "../datasets/plantvillage"
MODEL_SAVE_PATH = "../models/disease_model.pt"
CLASSES_SAVE_PATH = "../models/disease_classes.txt"

# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(DATA_DIR)
class_names = full_dataset.classes
print(f"Total classes: {len(class_names)}")
print(f"Total images: {len(full_dataset)}")
print(f"First 5 classes: {class_names[:5]}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Total classes: 47
Total images: 58541
First 5 classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [2]:
from torch.utils.data import Dataset, DataLoader, random_split

# Custom wrapper so train and validation can have independent transforms
class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset.dataset[self.subset.indices[idx]]

        if self.transform:
            img = self.transform(img)

        return img, label


# Reproducible split
generator = torch.Generator().manual_seed(42)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_raw, val_raw = random_split(
    full_dataset,
    [train_size, val_size],
    generator=generator
)

# Independent transforms
train_dataset = SubsetWithTransform(
    train_raw,
    transform=train_transform
)

val_dataset = SubsetWithTransform(
    val_raw,
    transform=val_transform
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Load pretrained ResNet18
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, len(class_names))
model = model.to(device)

print(f"\nModel loaded. Output classes: {len(class_names)}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=0.5
)

print("Ready to train.")

Training images: 46832
Validation images: 11709
Batches per epoch: 1464

Model loaded. Output classes: 47
Ready to train.


In [3]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [5]:
NUM_EPOCHS = 5
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

        # Print progress every 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {train_loss/(batch_idx+1):.3f} | "
                  f"Train Acc: {100.*train_correct/train_total:.1f}%")

    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    train_acc = 100. * train_correct / train_total

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1} COMPLETE")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Val Accuracy:   {val_acc:.2f}%")
    print(f"Val Loss:       {val_loss/len(val_loader):.3f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"New best model saved! Val Acc: {val_acc:.2f}%")

    scheduler.step()
    print(f"{'='*60}\n")

print(f"Training complete. Best validation accuracy: {best_val_acc:.2f}%")

# Save class names
with open(CLASSES_SAVE_PATH, 'w') as f:
    for cls in class_names:
        f.write(cls + '\n')
print(f"Class names saved to {CLASSES_SAVE_PATH}")

Epoch 1/5 | Batch 100/1464 | Loss: 0.578 | Train Acc: 81.0%
Epoch 1/5 | Batch 200/1464 | Loss: 0.538 | Train Acc: 82.3%
Epoch 1/5 | Batch 300/1464 | Loss: 0.505 | Train Acc: 83.5%
Epoch 1/5 | Batch 400/1464 | Loss: 0.483 | Train Acc: 84.1%
Epoch 1/5 | Batch 500/1464 | Loss: 0.461 | Train Acc: 84.8%
Epoch 1/5 | Batch 600/1464 | Loss: 0.448 | Train Acc: 85.3%
Epoch 1/5 | Batch 700/1464 | Loss: 0.428 | Train Acc: 85.9%
Epoch 1/5 | Batch 800/1464 | Loss: 0.411 | Train Acc: 86.4%
Epoch 1/5 | Batch 900/1464 | Loss: 0.395 | Train Acc: 86.9%
Epoch 1/5 | Batch 1000/1464 | Loss: 0.385 | Train Acc: 87.3%
Epoch 1/5 | Batch 1100/1464 | Loss: 0.374 | Train Acc: 87.6%
Epoch 1/5 | Batch 1200/1464 | Loss: 0.364 | Train Acc: 88.0%
Epoch 1/5 | Batch 1300/1464 | Loss: 0.356 | Train Acc: 88.2%
Epoch 1/5 | Batch 1400/1464 | Loss: 0.351 | Train Acc: 88.4%

EPOCH 1 COMPLETE
Train Accuracy: 88.51%
Val Accuracy:   90.94%
Val Loss:       0.283
New best model saved! Val Acc: 90.94%

Epoch 2/5 | Batch 100/1464 | L

In [6]:
# Quick sanity check -- load model and predict one image
from PIL import Image
import torch

# Reload class names
with open(CLASSES_SAVE_PATH, 'r') as f:
    loaded_classes = [line.strip() for line in f.readlines()]

# Load the saved model
loaded_model = models.resnet18(weights=None)
loaded_model.fc = nn.Linear(512, len(loaded_classes))
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Pick one test image from the dataset
import random
test_class = random.choice(loaded_classes)
test_folder = os.path.join(DATA_DIR, test_class)
test_image_path = os.path.join(test_folder, random.choice(os.listdir(test_folder)))

# Run prediction
img = Image.open(test_image_path).convert('RGB')
tensor = val_transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = loaded_model(tensor)
    probs = torch.softmax(output, dim=1)
    top_prob, top_idx = probs.max(1)

print(f"Test image from class : {test_class}")
print(f"Predicted class       : {loaded_classes[top_idx.item()]}")
print(f"Confidence            : {top_prob.item()*100:.1f}%")
print(f"Correct               : {test_class == loaded_classes[top_idx.item()]}")

Test image from class : Wheat___septoria
Predicted class       : Wheat___septoria
Confidence            : 98.6%
Correct               : True


C:\Users\dhari\AppData\Local\Temp\ipykernel_29972\348065357.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map